## Bước 1 — Chuẩn bị công cụ và tìm đúng file

`pandas` đọc bảng; `scikit-learn` xử lý dữ liệu và học mô hình;
`matplotlib` vẽ biểu đồ. Trên Kaggle thường đã có các thư viện này.
Nếu chạy trên máy cá nhân, dùng môi trường Python đã chuẩn bị trong thư mục dự án.

Notebook tự tìm file cạnh nó, trong thư mục `churn/`, hoặc trong dữ liệu đầu vào Kaggle.
Nó kiểm tra có cột `churn_3m` để không chọn nhầm train của Cafe.


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import json
import re
import sys
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay)
from threadpoolctl import threadpool_limits

SEED = 42
candidates = [Path("train.csv"), Path("churn/train.csv")]
if Path("/kaggle/input").exists():
    candidates += sorted(Path("/kaggle/input").rglob("train.csv"))
matches = []
for path in candidates:
    if path.is_file() and "churn_3m" in pd.read_csv(path, nrows=0).columns:
        if path.resolve() not in matches:
            matches.append(path.resolve())
if len(matches) != 1:
    raise ValueError(f"Cần đúng một train.csv của Churn; tìm được: {matches}")
DATA_FILE = matches[0]
OUTPUT = (Path("/kaggle/working/churn_outputs") if Path("/kaggle/working").exists()
          else DATA_FILE.parent / "outputs_train_only")
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Dữ liệu:", DATA_FILE)
print("Kết quả sẽ lưu:", OUTPUT)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__,
      "| scikit-learn:", sklearn.__version__)


## Bước 2 — Đọc bảng và hiểu cột đáp án

`shape` trả về (số dòng, số cột). `head()` cho xem 5 dòng đầu.
`customer_id` là mã nhận diện, không dùng làm đặc trưng cho mô hình trong bài này.
`churn_3m` là nhãn thật để mô hình học; tuyệt đối không đưa nó vào các cột đầu vào.

Ví dụ: mô hình nhìn tuổi, thu nhập, số giao dịch… rồi dự đoán khả năng đóng thẻ.
Sau đó ta so dự đoán với `churn_3m` đã biết.


In [ ]:
raw = pd.read_csv(DATA_FILE)
print("Số dòng và số cột:", raw.shape)
display(raw.head())
raw.info()
assert raw["churn_3m"].notna().all(), "Có nhãn bị thiếu; cần kiểm tra trước khi học."
assert set(raw["churn_3m"].unique()) == {0, 1}, "Nhãn phải gồm hai giá trị 0 và 1."



## Bước 3 — Bỏ bản ghi trùng hoàn toàn

Nếu cùng một khách hàng giống hệt xuất hiện ở cả tập học và tập kiểm tra,
điểm có thể cao hơn thực tế vì mô hình đã gặp bản ghi đó.
Ta chỉ bỏ dòng **trùng toàn bộ cột**. Nếu một ID còn nhiều dòng khác nhau,
notebook sẽ dừng để kiểm tra ý nghĩa dữ liệu thay vì tự chọn một dòng.


In [ ]:
duplicate_count = int(raw.duplicated().sum())
data = raw.drop_duplicates().reset_index(drop=True)
assert data["customer_id"].notna().all(), "Có mã khách hàng bị thiếu."
assert data["customer_id"].is_unique, "Một ID có nhiều bản ghi khác nhau; cần kiểm tra."
print("Dòng ban đầu:", len(raw))
print("Dòng trùng hoàn toàn đã bỏ:", duplicate_count)
print("Khách hàng còn lại:", len(data))


## Bước 4 — Chia phần học và phần kiểm tra

Hãy hình dung 80% là **bài tập có đáp án để ôn**, 20% là **bài kiểm tra cất riêng**.
Cả hai đều lấy từ `train.csv`; phần giữ riêng không phải `test.csv` chính thức.

- `X`: thông tin khách hàng, đã bỏ ID và đáp án.
- `y`: đáp án `churn_3m`.
- `stratify=y`: giữ tỷ lệ hai nhóm gần giống nhau ở hai phần.
- `random_state=42`: giúp lần chạy sau chia giống lần trước.

Ta chỉ chọn mô hình bằng 80% đầu, sau đó mới đo kết quả trên 20% giữ riêng.
Đây là cách chia khởi đầu cho bảng khách hàng tại một thời điểm; chưa chứng minh khả năng dự đoán ở thời điểm tương lai khác.


In [ ]:
X = data.drop(columns=["customer_id", "churn_3m"])
y = data["churn_3m"].astype(int)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
assert set(X_train.index).isdisjoint(X_holdout.index)
assert "customer_id" not in X_train and "churn_3m" not in X_train
split_summary = pd.DataFrame({
    "phần": ["Học và chọn mô hình", "Giữ riêng để kiểm tra"],
    "số khách": [len(X_train), len(X_holdout)],
    "tỷ lệ đóng thẻ": [y_train.mean(), y_holdout.mean()],
})
display(split_summary)


## Bước 5 — Khám phá phần học

Khảo sát 80% dùng để học: nhãn có cân bằng không, cột nào thiếu nhiều,
có các cách viết khác nhau cho cùng một ý nghĩa không?
Biểu đồ và các bảng sau là tư liệu cho báo cáo.

Một mô hình luôn nói “không đóng thẻ” có thể đúng gần 90% nhưng bỏ sót toàn bộ
khách đóng thẻ. Vì thế bài này dùng **ROC AUC** làm chỉ số chính, không chọn theo accuracy.


In [ ]:
class_counts = y_train.value_counts().reindex([0, 1], fill_value=0)
display(pd.DataFrame({"số khách": class_counts, "tỷ lệ": class_counts / len(y_train)}))
missing = X_train.isna().mean().sort_values(ascending=False).rename("tỷ lệ thiếu")
display(missing[missing > 0].to_frame())
display(X_train.describe().T)
for column in ["gender", "province", "occupation", "currency"]:
    print("\n", column)
    display(X_train[column].value_counts(dropna=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].bar(["No churn (0)", "Churn (1)"], class_counts.values,
            color=["#2563eb", "#ea580c"])
axes[0].set(title="Training labels", ylabel="Customers")
missing.head(10).sort_values().plot.barh(ax=axes[1], color="#2563eb")
axes[1].set(title="Missing values in training features", xlabel="Fraction missing")
fig.savefig(OUTPUT / "01_data_overview.png", dpi=150)
plt.show()


## Bước 6 — Tạo mô hình mốc ban đầu (baseline)

Bắt đầu giống ý tưởng starter: chỉ dùng cột số và Logistic Regression.
Tên có chữ “Regression” nhưng mô hình này dùng để **phân loại**.

`Pipeline` thực hiện tuần tự: điền ô thiếu bằng trung vị → chuẩn hóa → học mô hình.
Các giá trị trung vị và tham số chuẩn hóa chỉ được học từ phần huấn luyện ở mỗi lần chia,
giúp tránh dùng thông tin của phần kiểm tra trước khi chấm điểm.

Điểm baseline ở đây không phải bản sao chính xác điểm starter:
ta đã bỏ dòng trùng, giữ riêng holdout và đưa điền thiếu vào pipeline.


In [ ]:
numeric_steps = Pipeline([
    ("fill_missing", SimpleImputer(strategy="median", add_indicator=True,
                                   keep_empty_features=True)),
    ("scale", StandardScaler()),
])
numeric_only = ColumnTransformer([
    ("numbers", clone(numeric_steps), make_column_selector(dtype_include=np.number)),
], remainder="drop")
baseline = Pipeline([
    ("prepare", numeric_only),
    ("model", LogisticRegression(max_iter=3000, random_state=SEED)),
])
print("Đã tạo baseline; chưa chấm trên phần holdout.")


## Bước 7 — Làm sạch và tạo đặc trưng

Các quy tắc dưới đây không học trung bình hay thống kê từ holdout:

- Gộp cách viết tương đương, ví dụ `F`, `female`, `Nữ` và các tên tỉnh viết khác nhau.
- Đọc ngày theo ba định dạng tường minh: YYYY-MM-DD, DD/MM/YYYY, YYYYMMDD.
  Giả định ngày có dấu `/` theo ngày/tháng/năm; cần xác nhận nếu thầy có mô tả khác.
- Tính số ngày mở thẻ đến **30/06/2026**, ngày chốt được ghi trong starter đã cung cấp.
- Giả định tuổi ngoài 18–100 cần kiểm tra: đánh dấu và để thiếu để pipeline xử lý.
  Đây là quy tắc thực nghiệm, không phải kết luận mọi khách ngoài khoảng này đều sai.
- Tỷ lệ sử dụng âm được đánh dấu và để thiếu; tỷ lệ lớn hơn 1 được giữ lại,
  vì chưa đủ căn cứ tự đổi sang phần trăm. Giá trị giao dịch âm cũng được giữ lại.
- Thu nhập/hạn mức có giá trị ở nhiều thang đo. **Chưa tự đổi đơn vị khi chưa có mô tả**;
  tạo phép biến đổi log có giữ dấu để giảm ảnh hưởng chênh lệch lớn. Cách này không sửa được lỗi đơn vị.

Các giả định này phải được ghi trong báo cáo và có thể thử thay đổi bằng cross-validation.


In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower().replace("đ", "d")
    text = "".join(c for c in unicodedata.normalize("NFD", text)
                   if unicodedata.category(c) != "Mn")
    return " ".join(text.split())

def build_features(frame):
    result = frame.copy()
    for col in result.select_dtypes(exclude=np.number).columns:
        normalized = result[col].map(normalize_text)
        result[col] = normalized.astype(object).where(normalized.notna(), np.nan)

    result["gender"] = result["gender"].replace({
        "f": "female", "nu": "female", "m": "male", "nam": "male"
    })
    result["occupation"] = result["occupation"].map(
        lambda v: re.sub(r"[\s_-]+", "_", v) if isinstance(v, str) else v
    )
    result["province"] = result["province"].replace({
        "hn": "ha noi", "hanoi": "ha noi", "hcm": "ho chi minh",
        "tp.hcm": "ho chi minh", "tp hcm": "ho chi minh",
        "danang": "da nang", "br-vt": "ba ria - vung tau",
    })

    date_text = result.pop("open_date").astype("string")
    opened = pd.Series(pd.NaT, index=result.index, dtype="datetime64[ns]")
    for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%Y%m%d"]:
        opened = opened.fillna(pd.to_datetime(date_text, format=fmt, errors="coerce"))
    snapshot = pd.Timestamp("2026-06-30")
    future = opened > snapshot
    result["open_date_invalid"] = (opened.isna() | future).astype(float)
    opened = opened.mask(future)
    result["tenure_days"] = (snapshot - opened).dt.days.astype(float)
    result["open_month"] = opened.dt.month.astype(float)

    invalid_age = result["age"].notna() & ~result["age"].between(18, 100)
    result["age_flagged"] = invalid_age.astype(float)
    result["age"] = result["age"].mask(invalid_age)
    negative_utilization = result["utilization_6m"] < 0
    result["utilization_negative"] = negative_utilization.astype(float)
    result["utilization_6m"] = result["utilization_6m"].mask(negative_utilization)

    for col in ["income_month", "credit_limit", "txn_amount_3m"]:
        values = result.pop(col)
        result[col + "_signed_log"] = np.sign(values) * np.log1p(values.abs())
    return result

feature_example = build_features(X_train.head())
display(feature_example)
print("Ví dụ số cột trước/sau tạo đặc trưng:", X_train.shape[1], feature_example.shape[1])


## Bước 8 — Thêm cột dạng chữ và mô hình thứ hai

Mô hình cần số, nên `OneHotEncoder` đổi nhóm chữ thành cột 0/1.
Ví dụ giới tính được thể hiện bằng các cột chỉ báo, không gán tùy ý “nam = 1, nữ = 2”.
Nhãn hiếm và nhãn mới được bộ mã hóa xử lý mà không cần nhìn trước phần kiểm tra.

Ta so sánh ba phiên bản: baseline số; Logistic Regression có làm sạch và thêm thông tin;
mô hình cây tăng cường `HistGradientBoostingClassifier`, có thể học quan hệ phi tuyến.
Chưa biết phiên bản nào tốt hơn cho đến khi đo bằng cùng các fold.


In [ ]:
categorical_steps = Pipeline([
    ("fill_missing", SimpleImputer(strategy="constant", fill_value="missing",
                                   keep_empty_features=True)),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                             min_frequency=10)),
])
all_columns = ColumnTransformer([
    ("numbers", clone(numeric_steps), make_column_selector(dtype_include=np.number)),
    ("categories", categorical_steps, make_column_selector(dtype_exclude=np.number)),
])

def improved_pipeline(estimator):
    return Pipeline([
        ("features", FunctionTransformer(build_features, validate=False)),
        ("prepare", clone(all_columns)),
        ("model", estimator),
    ])

models = {
    "01_numeric_logistic": baseline,
    "02_cleaned_logistic": improved_pipeline(
        LogisticRegression(max_iter=3000, random_state=SEED)),
    "03_cleaned_gradient_boosting": improved_pipeline(
        HistGradientBoostingClassifier(
            max_iter=150, learning_rate=0.07, max_leaf_nodes=17,
            min_samples_leaf=35, l2_regularization=10.0,
            early_stopping=True, scoring="roc_auc", random_state=SEED)),
}
print("Các phiên bản:", list(models))


## Bước 9 — So sánh bằng cross-validation trên phần học

Chia 80% phần học thành 5 phần nhỏ: mỗi lượt học trên 4 phần, chấm trên 1 phần.
Lặp đủ 5 lượt rồi lấy trung bình. Tất cả mô hình dùng **cùng 5 fold**.

**ROC AUC:** đo khả năng xếp khách đóng thẻ cao hơn khách không đóng thẻ;
khoảng 0,5 tương đương xếp ngẫu nhiên, 1,0 là phân biệt hoàn hảo trên dữ liệu được chấm.
AUC 0,80 không có nghĩa dự đoán đúng 80% khách hàng.

**Average Precision (AP):** bổ sung góc nhìn cho nhóm đóng thẻ ít gặp; càng cao càng tốt.
Dùng ROC AUC trung bình để chọn mô hình, không nhìn điểm holdout để chọn.
Độ lệch chuẩn giữa các fold chỉ thể hiện dao động, không phải khoảng tin cậy.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_folds = list(cv.split(X_train, y_train))
rows = []
fold_rows = []
for name, model in models.items():
    print("Đang chạy:", name, flush=True)
    with threadpool_limits(limits=2):
        scores = cross_validate(
            model, X_train, y_train, cv=cv_folds,
            scoring={"auc": "roc_auc", "ap": "average_precision"},
            n_jobs=1, error_score="raise",
        )
    rows.append({"model": name,
                 "cv_auc_mean": scores["test_auc"].mean(),
                 "cv_auc_std": scores["test_auc"].std(ddof=1),
                 "cv_ap_mean": scores["test_ap"].mean()})
    for fold, (auc, ap) in enumerate(zip(scores["test_auc"], scores["test_ap"]), 1):
        fold_rows.append({"model": name, "fold": fold, "auc": auc, "ap": ap})
comparison = pd.DataFrame(rows).sort_values("cv_auc_mean", ascending=False).reset_index(drop=True)
best_name = comparison.iloc[0]["model"]
display(comparison.round(4))
print("Chọn theo CV:", best_name)
comparison.to_csv(OUTPUT / "model_comparison.csv", index=False)
pd.DataFrame(fold_rows).to_csv(OUTPUT / "cv_folds.csv", index=False)


In [ ]:
probe = clone(models["03_cleaned_gradient_boosting"])
probe.fit(X_train, y_train)
print("Số vòng đã học:", probe.named_steps["model"].n_iter_)

## Bước 10 — Kiểm tra một lần mô hình đã chọn

Huấn luyện mô hình thắng CV bằng toàn bộ 80% phần học, rồi dự đoán 20% giữ riêng.
`predict_proba(...)[:, 1]` lấy xác suất của nhãn 1: đóng thẻ.
Đây là điểm **holdout nội bộ**, chưa phải kết quả Kaggle.

Để minh họa nhãn dự đoán, dùng ngưỡng 0,5: xác suất từ 0,5 trở lên → dự đoán đóng thẻ.
ROC AUC và AP dùng điểm xác suất, không cần ngưỡng này.
Ngưỡng 0,5 chưa chắc phù hợp cho ứng dụng thực tế; không tối ưu nó bằng holdout.


In [ ]:
final_model = clone(models[best_name])
with threadpool_limits(limits=2):
    final_model.fit(X_train, y_train)
    probabilities = final_model.predict_proba(X_holdout)[:, 1]
assert len(probabilities) == len(y_holdout)
assert np.isfinite(probabilities).all()
assert ((probabilities >= 0) & (probabilities <= 1)).all()
predictions = (probabilities >= 0.5).astype(int)

metrics = {
    "selected_model": best_name,
    "holdout_roc_auc": float(roc_auc_score(y_holdout, probabilities)),
    "holdout_average_precision": float(average_precision_score(y_holdout, probabilities)),
    "accuracy_at_0_5": float(accuracy_score(y_holdout, predictions)),
    "precision_at_0_5": float(precision_score(y_holdout, predictions, zero_division=0)),
    "recall_at_0_5": float(recall_score(y_holdout, predictions, zero_division=0)),
    "f1_at_0_5": float(f1_score(y_holdout, predictions, zero_division=0)),
    "always_no_churn_accuracy": float((y_holdout == 0).mean()),
    "constant_score_roc_auc": 0.5,
    "constant_score_average_precision": float(y_holdout.mean()),
    "rows_raw": len(raw), "duplicates_removed": duplicate_count,
    "rows_unique": len(data), "train_rows": len(X_train),
    "holdout_rows": len(X_holdout), "seed": SEED,
}
display(pd.Series(metrics).to_frame("kết quả"))
cm = confusion_matrix(y_holdout, predictions, labels=[0, 1])
print("Ma trận nhầm lẫn [[TN, FP], [FN, TP]]:")
print(cm)
metrics["confusion_matrix_labels_0_1"] = cm.tolist()


## Bước 11 — Xem biểu đồ và hiểu các loại lỗi

- **Precision:** trong số khách dự đoán đóng thẻ, bao nhiêu người thực sự đóng?
- **Recall:** trong số khách thực sự đóng thẻ, mô hình tìm ra được bao nhiêu?
- **False negative (FN):** khách sẽ đóng thẻ nhưng mô hình bỏ sót.
- **False positive (FP):** khách không đóng thẻ nhưng mô hình báo sẽ đóng.

Nhìn cả AUC/AP và lỗi ở ngưỡng 0,5. Một mô hình xếp hạng khá vẫn có thể bỏ sót
nhiều khách đóng thẻ ở ngưỡng này. Không diễn giải các điểm xác suất như đã được
kiểm chứng độ hiệu chỉnh, và không kết luận đặc trưng nào gây ra việc đóng thẻ.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
RocCurveDisplay.from_predictions(y_holdout, probabilities, ax=axes[0], name="Selected model")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Random ranking")
axes[0].set_title("Holdout ROC")
axes[0].legend(loc="lower right")
PrecisionRecallDisplay.from_predictions(y_holdout, probabilities, ax=axes[1], name="Selected model")
axes[1].axhline(y_holdout.mean(), linestyle="--", color="gray", label="Class prevalence")
axes[1].set_title("Holdout precision-recall")
axes[1].legend(loc="upper right")
ConfusionMatrixDisplay(cm, display_labels=["No churn", "Churn"]).plot(
    ax=axes[2], colorbar=False, cmap="Blues", values_format="d")
axes[2].set_title("Threshold = 0.5")
fig.savefig(OUTPUT / "02_holdout_evaluation.png", dpi=150)
plt.show()


## Bước 12 — Lưu kết quả để viết báo cáo

File `holdout_predictions_NOT_submission.csv` chứa cả nhãn thật và dự đoán của phần
kiểm tra nội bộ, **không phải file nộp Kaggle**.
Các file kết quả được tạo từ lần chạy này, không dùng số liệu giả.

Mô hình hiện chỉ học trên 80% dữ liệu để còn kiểm tra độc lập.
Khi có test chính thức, giữ lại phương pháp đã chọn, học lại trên toàn bộ train đã
xử lý trùng, dự đoán test rồi tạo submission đúng định dạng cuộc thi.


In [ ]:
holdout_output = pd.DataFrame({
    "customer_id": data.loc[X_holdout.index, "customer_id"].to_numpy(),
    "actual_churn_3m": y_holdout.to_numpy(),
    "predicted_probability": probabilities,
    "predicted_label_at_0_5": predictions,
    "partition": "internal_holdout_not_kaggle_test",
})
assert holdout_output["customer_id"].is_unique
holdout_output.to_csv(OUTPUT / "holdout_predictions_NOT_submission.csv", index=False)
metrics["source_file"] = DATA_FILE.name
metrics["source_sha256"] = __import__("hashlib").sha256(DATA_FILE.read_bytes()).hexdigest()
metrics["package_versions"] = {"python": sys.version.split()[0],
                               "numpy": np.__version__, "pandas": pd.__version__,
                               "scikit-learn": sklearn.__version__}
(OUTPUT / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")

table_lines = ["| Phiên bản | CV ROC AUC trung bình | Độ lệch chuẩn | CV AP |",
               "|---|---:|---:|---:|"]
for row in comparison.itertuples(index=False):
    table_lines.append(f"| {row.model} | {row.cv_auc_mean:.4f} | {row.cv_auc_std:.4f} | {row.cv_ap_mean:.4f} |")
result_table = "\n".join(table_lines)
report = f"""# Churn — kết quả thực nghiệm chỉ dùng train.csv

Bản ghi tự động từ lần chạy notebook. Chưa có điểm Kaggle.

## Dữ liệu và cách đánh giá

- File có {len(raw):,} dòng và {raw.shape[1]} cột.
- Bỏ {duplicate_count:,} dòng trùng hoàn toàn, còn {len(data):,} khách hàng có ID duy nhất.
- Tỷ lệ đóng thẻ sau bỏ trùng: {y.mean():.2%}.
- Chia có giữ tỷ lệ nhãn: {len(X_train):,} khách để học/chọn mô hình, {len(X_holdout):,} khách để kiểm tra cuối.
- So sánh ba pipeline bằng cùng 5 fold trên phần học; chọn theo ROC AUC trung bình.
- Median, chuẩn hóa và mã hóa đều được fit bên trong từng fold.

## So sánh nội bộ trên phần học

{result_table}

## Kiểm tra mô hình đã chọn

- Mô hình: **{best_name}**.
- Holdout ROC AUC: **{metrics['holdout_roc_auc']:.4f}**.
- Holdout Average Precision: **{metrics['holdout_average_precision']:.4f}**.
- Ở ngưỡng 0,5: precision {metrics['precision_at_0_5']:.4f}, recall {metrics['recall_at_0_5']:.4f}, F1 {metrics['f1_at_0_5']:.4f}.
- Accuracy {metrics['accuracy_at_0_5']:.4f}; chỉ đoán không đóng thẻ đã đạt accuracy {metrics['always_no_churn_accuracy']:.4f}.
- Ma trận [[TN, FP], [FN, TP]]: {cm.tolist()}.
- AUC của điểm hằng là 0,5; AP của điểm hằng bằng tỷ lệ đóng thẻ của holdout ({y_holdout.mean():.4f}).

## Giả định và giới hạn

- Tuổi ngoài 18–100 được đánh dấu và để thiếu; utilization âm được đánh dấu và để thiếu. Cần xác nhận ý nghĩa với mô tả dữ liệu.
- Ngày dạng dấu gạch chéo được hiểu theo DD/MM/YYYY. Mốc tính tuổi thẻ 30/06/2026 lấy từ starter.
- Không tự quy đổi các giá trị tiền nghi lẫn đơn vị; signed log chỉ giảm chênh lệch thang đo. Tỷ lệ sử dụng lớn hơn 1 và tiền giao dịch âm chưa bị tự sửa.
- Kết quả là ước lượng trên một cách chia khách hàng; chưa đánh giá trên test Kaggle hay một thời điểm tương lai khác.
- Điểm CV chênh lệch nhỏ không đủ để khẳng định mô hình tốt hơn một cách chắc chắn.
- Holdout chỉ dùng sau chọn mô hình. Nếu tiếp tục cải tiến, dùng CV, không tối ưu theo holdout này.
- Chưa hiệu chỉnh xác suất hoặc tối ưu ngưỡng quyết định.

## File kết quả

- model_comparison.csv: bảng so sánh CV.
- cv_folds.csv: điểm từng fold.
- metrics.json: điểm và phiên bản thư viện, mã băm nguồn dữ liệu.
- holdout_predictions_NOT_submission.csv: dự đoán kiểm tra nội bộ, không dùng nộp Kaggle.
- 01_data_overview.png và 02_holdout_evaluation.png: biểu đồ cho báo cáo.
"""
# Bỏ phần thụt đầu dòng của nội dung Markdown được tạo trong ô code.
report = "\n".join(line[4:] if line.startswith("    ") else line for line in report.splitlines())
(OUTPUT / "KET_QUA_CHURN.md").write_text(report, encoding="utf-8")
print("Đã lưu các kết quả tại:", OUTPUT)
print("Chưa tạo submission; notebook này không đọc test.csv.")


In [ ]:
from sklearn.metrics import precision_recall_curve

# 1. Tạo xác suất ngoài fold trên 9.600 khách phần học
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve
from sklearn.base import clone
from threadpoolctl import threadpool_limits
from threadpoolctl import threadpool_limits
with threadpool_limits(limits=2):
    oof_probability = cross_val_predict(
        clone(models[best_name]),
        X_train,
        y_train,
        cv=cv_folds,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

# 2. Tìm ngưỡng cho F1 cao nhất trên các dự đoán ngoài fold
precision, recall, thresholds = precision_recall_curve(
    y_train, oof_probability
)

f1_by_threshold = np.divide(
    2 * precision[:-1] * recall[:-1],
    precision[:-1] + recall[:-1],
    out=np.zeros_like(thresholds),
    where=(precision[:-1] + recall[:-1]) > 0
)
# Chỉ xét các ngưỡng có precision OOF đạt ít nhất 70%
minimum_precision = 0.70
eligible_indices = np.flatnonzero(
    precision[:-1] >= minimum_precision
)

if len(eligible_indices) == 0:
    raise ValueError(
        "Không có ngưỡng nào đạt precision OOF ít nhất 70%."
    )

# Trong các ngưỡng đạt yêu cầu, chọn ngưỡng có F1 cao nhất
best_index = eligible_indices[
    np.argmax(f1_by_threshold[eligible_indices])
]

chosen_threshold = float(thresholds[best_index])

print("Precision OOF tại ngưỡng chọn:",
      round(float(precision[best_index]), 4))
print("Recall OOF tại ngưỡng chọn:",
      round(float(recall[best_index]), 4))